# Phase 1 Final Claim-State Closeout

This notebook is orchestration only. It records the final Phase 1 claim-state disposition from a reviewed final governance reconciliation run.

Scientific integrity rules:

- This notebook does not train models, recompute metrics, edit thresholds, or repair failed governance surfaces.
- It preserves the final governance blockers exactly as observed.
- A fail-closed closeout is a valid scientific outcome when controls, calibration, or influence fail.
- The output must explicitly keep headline Phase 1 claims closed.
- Any remediation after this closeout requires preregistration/revision-policy review; it must not be done by post-hoc threshold changes.


In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import json
import os
import subprocess
import sys

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')

def run(cmd, cwd=None, check=True):
    display = []
    for item in cmd:
        text = str(item)
        if 'Authorization: Basic' in text:
            display.append('http.extraHeader=<redacted>')
        else:
            display.append(text)
    print('$ ' + ' '.join(display))
    result = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, check=check)
    return result

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    header = 'Authorization: Basic ' + base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
unit_result = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=str(REPO_DIR), text=True)
if unit_result.returncode != 0:
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])


In [ ]:
# Cell 2 - Select reviewed governance source and keep launch disabled by default.

from pathlib import Path
import hashlib
import json

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Pin the reviewed post-dedicated-controls governance reconciliation run.
GOVERNANCE_RECONCILIATION_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase1_final_governance_reconciliation/20260422T082255821648Z')

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_claim_state_closeout'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_claim_state_closeout_plan'

RUN_FINAL_CLAIM_STATE_CLOSEOUT = False
REQUIRED_ACK = 'I reviewed final claim-state closeout and understand Phase 1 claims remain closed because governance blockers remain'
MANUAL_LAUNCH_ACK = ''

FIX_MARKER = 'phase1_final_claim_state_closeout_v1_20260422'
print('Notebook fix marker:', FIX_MARKER)


In [ ]:
# Cell 3 - Resolve paths and validate prereg plus governance claim boundary.

def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def resolve_latest(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        return Path(latest.read_text(encoding='utf-8').strip())
    runs = sorted([p for p in root.iterdir() if p.is_dir()]) if root.exists() else []
    if not runs:
        raise FileNotFoundError(f'No runs found under {root}')
    return runs[-1]

def resolve_prereg_bundle(path: Path) -> Path:
    path = Path(path)
    if path.exists():
        return path
    candidates = sorted(ARTIFACT_ROOT.glob('prereg/*/prereg_bundle.json'))
    print('Configured prereg bundle not found:', path)
    print('Found prereg bundles:', len(candidates))
    for item in candidates[-5:]:
        print('candidate:', item)
    if not candidates:
        raise FileNotFoundError(f'No prereg_bundle.json found under {ARTIFACT_ROOT / "prereg"}')
    return candidates[-1]

PREREG_BUNDLE = resolve_prereg_bundle(Path(PREREG_BUNDLE))
GOVERNANCE_RECONCILIATION_RUN = Path(GOVERNANCE_RECONCILIATION_RUN) if GOVERNANCE_RECONCILIATION_RUN else resolve_latest(ARTIFACT_ROOT / 'phase1_final_governance_reconciliation')

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

gov_summary = read_json(GOVERNANCE_RECONCILIATION_RUN / 'phase1_final_governance_reconciliation_summary.json')
gov_claim_state = read_json(GOVERNANCE_RECONCILIATION_RUN / 'phase1_final_governance_claim_state.json')
assert gov_summary.get('comparator_outputs_complete') is True
assert gov_summary.get('runtime_logs_audited_for_all_required_comparators') is True
assert gov_summary.get('claim_ready') is False
assert gov_summary.get('headline_phase1_claim_open') is False
assert gov_summary.get('full_phase1_claim_bearing_run_allowed') is False
assert gov_claim_state.get('claim_ready') is False
print('Governance reconciliation run:', GOVERNANCE_RECONCILIATION_RUN)
print('Governance status:', gov_summary.get('status'))
print('Governance surfaces:', gov_summary.get('governance_surfaces'))
print('Claim blockers:', gov_summary.get('claim_blockers'))


In [ ]:
# Cell 4 - Preflight runner/config availability and write a reviewable plan artifact.

from datetime import datetime, timezone

sys.path.insert(0, str(REPO_DIR))
preflight_blockers = []
try:
    from src.phase1.final_claim_state_closeout import run_phase1_final_claim_state_closeout  # noqa: F401
    runner_available = True
except Exception as exc:
    runner_available = False
    preflight_blockers.append(f'final claim-state closeout runner import failed: {exc}')

required_repo_files = [
    REPO_DIR / 'src/phase1/final_claim_state_closeout.py',
    REPO_DIR / 'configs/phase1/final_claim_state_closeout.json',
]
for path in required_repo_files:
    if not path.exists():
        preflight_blockers.append(f'missing repo file: {path.relative_to(REPO_DIR)}')

PLAN_ROOT.mkdir(parents=True, exist_ok=True)
plan_dir = PLAN_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir.mkdir(parents=True, exist_ok=False)

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_claim_state_closeout',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--governance-reconciliation-run', str(GOVERNANCE_RECONCILIATION_RUN),
    '--output-root', str(OUTPUT_ROOT),
    '--closeout-config', 'configs/phase1/final_claim_state_closeout.json',
]

plan = {
    'status': 'phase1_final_claim_state_closeout_plan_recorded',
    'created_utc': plan_dir.name,
    'fix_marker': FIX_MARKER,
    'runner_available': runner_available,
    'run_flag': RUN_FINAL_CLAIM_STATE_CLOSEOUT,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight_blockers,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'governance_reconciliation_run': str(GOVERNANCE_RECONCILIATION_RUN),
    'output_root': str(OUTPUT_ROOT),
    'command': cmd,
    'scientific_integrity_rule': 'Plan only. Closeout records failure/blockers and must not open claims.',
}
(plan_dir / 'phase1_final_claim_state_closeout_plan.json').write_text(json.dumps(plan, indent=2) + '\n', encoding='utf-8')
print(json.dumps({k: plan[k] for k in ['status', 'runner_available', 'run_flag', 'ack_matched', 'preflight_blockers']}, indent=2))
if preflight_blockers:
    raise RuntimeError(preflight_blockers)


In [ ]:
# Cell 5 - Manual hold or launch the CLI-backed closeout runner.

if not RUN_FINAL_CLAIM_STATE_CLOSEOUT:
    hold = {
        'status': 'phase1_final_claim_state_closeout_manual_hold',
        'run_flag': RUN_FINAL_CLAIM_STATE_CLOSEOUT,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'plan_dir': str(plan_dir),
    }
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch final closeout without explicit claim-closed acknowledgement.')
else:
    launch_review = {
        'status': 'phase1_final_claim_state_closeout_launch_review_passed',
        'run_flag': RUN_FINAL_CLAIM_STATE_CLOSEOUT,
        'ack_matched': True,
        'claim_ready_before_run': False,
    }
    (plan_dir / 'phase1_final_claim_state_closeout_launch_review.json').write_text(json.dumps(launch_review, indent=2) + '\n', encoding='utf-8')
    run(cmd, cwd=REPO_DIR)
    print('Final claim-state closeout command completed. Review generated artifacts before any revision planning.')


In [ ]:
# Cell 6 - Inspect latest final closeout output if execution was launched.

summary = None
latest_run = None
if RUN_FINAL_CLAIM_STATE_CLOSEOUT:
    latest_run = resolve_latest(OUTPUT_ROOT)
    print('Latest final claim-state closeout output:', latest_run)
    required_files = [
        'phase1_final_claim_state_closeout_inputs.json',
        'phase1_final_claim_state_closeout_summary.json',
        'phase1_final_claim_state_closeout_report.md',
        'phase1_final_claim_state_closeout_source_links.json',
        'phase1_final_claim_state_closeout_input_validation.json',
        'phase1_final_blocker_table.json',
        'phase1_final_claim_disposition.json',
        'phase1_final_revision_decision_memo.md',
        'phase1_final_claim_state_closeout_manifest.json',
    ]
    for name in required_files:
        print(name, 'exists =', (latest_run / name).exists())
    summary = read_json(latest_run / 'phase1_final_claim_state_closeout_summary.json')
    print(json.dumps({
        'status': summary.get('status'),
        'final_claim_blocked': summary.get('final_claim_blocked'),
        'blocking_surfaces': summary.get('blocking_surfaces'),
        'claims_opened': summary.get('claims_opened'),
        'revision_required_for_remediation': summary.get('revision_required_for_remediation'),
        'claim_blockers': summary.get('claim_blockers'),
    }, indent=2))
else:
    print('Manual hold only; no final closeout output inspected.')


In [ ]:
# Cell 7 - Assertions and non-claim review note.

if RUN_FINAL_CLAIM_STATE_CLOSEOUT:
    assert summary is not None
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    assert summary.get('claims_opened') is False
    assert summary.get('final_claim_blocked') is True
    disposition = read_json(latest_run / 'phase1_final_claim_disposition.json')
    blocker_table = read_json(latest_run / 'phase1_final_blocker_table.json')
    assert disposition.get('claims_opened') is False
    assert blocker_table.get('blocking_surfaces'), 'Closeout should record blocking governance surfaces.'
    review_note = {
        'status': 'phase1_final_claim_state_closeout_review_recorded',
        'reviewed_run': str(latest_run),
        'scope': 'final fail-closed Phase 1 claim-state disposition',
        'final_claim_blocked': summary.get('final_claim_blocked'),
        'blocking_surfaces': summary.get('blocking_surfaces'),
        'metrics_interpretation': {
            'allowed': 'Governance closeout and revision-decision context only.',
            'not_allowed': 'Do not use this closeout as evidence for decoder efficacy, privileged-transfer efficacy, or full Phase 1 performance.',
        },
        'next_allowed_steps': [
            'Archive this as the final fail-closed Phase 1 claim-state unless a prereg/revision-policy remediation is approved.',
            'Do not alter thresholds, omit failed controls/calibration/influence, or open headline claims from this run.',
        ],
        'not_ok_to_claim': disposition.get('not_ok_to_claim'),
    }
    (latest_run / 'phase1_final_claim_state_closeout_review_note.json').write_text(json.dumps(review_note, indent=2) + '\n', encoding='utf-8')
    print('Review note written:', latest_run / 'phase1_final_claim_state_closeout_review_note.json')
    print(json.dumps(review_note, indent=2))
else:
    print('No assertions beyond plan/manual hold because RUN_FINAL_CLAIM_STATE_CLOSEOUT is False.')


In [ ]:
# Cell 8 - Closeout.

print('================ Phase 1 Final Claim-State Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_FINAL_CLAIM_STATE_CLOSEOUT)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
print('Governance reconciliation run:', GOVERNANCE_RECONCILIATION_RUN)
print('Preflight blockers:', preflight_blockers)
if RUN_FINAL_CLAIM_STATE_CLOSEOUT:
    print('Latest final claim-state closeout output:', latest_run)
    print('Final claim blocked:', summary.get('final_claim_blocked'))
    print('Blocking surfaces:', summary.get('blocking_surfaces'))
    print('Claims opened:', summary.get('claims_opened'))
    print('Revision required for remediation:', summary.get('revision_required_for_remediation'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review phase1_final_revision_decision_memo.md before any future remediation planning.')
else:
    print('HELD: Runner appears available, but closeout was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit claim-closed acknowledgement if appropriate.')
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
